                YOUR PDFs
                   ↓
             DirectoryLoader
                   ↓
          PyPDFLoader reads PDFs
                   ↓
              Documents
                   ↓
          CharacterTextSplitter
                   ↓
               Chunks
                   ↓
         HuggingFace Embeddings
                   ↓
              Vectors
                   ↓
                Chroma
                   ↓
            Vector Database
            
So the overall purpose is:

Take PDFs → extract their text → break text into smaller pieces → convert those pieces into numerical vectors → store them in a vector database.

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement langchain-chromadb (from versions: none)
ERROR: No matching distribution found for langchain-chromadb


In [2]:
from langchain_community.document_loaders import DirectoryLoader, UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

C:\Users\Acer\AppData\Local\Temp\ipykernel_29228\3510866223.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, UnstructuredFileLoader


In [3]:
#Some document/text processing functionality requires language-processing resources, so the notebook downloads the required NLTK data
import nltk
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

To do:

Install Poppler
Install Tesseract  for scanning text from images 

In [4]:
# configuration
docs_dir_path = "./docs_dir"
vector_db_path = "./vector_db"  #where to store the vector database.
collection_name  = "document_collection" #name of the Chroma collection where your document vectors are stored

In [5]:
# loading the embedding model
embedding = HuggingFaceEmbeddings()  #converting text into numbers that capture meaning.

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 74c9c718-2a68-48d7-a710-97c9e2a5f0b5)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 7ac7fa09-e6d1-4082-a9c1-cee8f79859b8)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/modules.json
Retrying in 1s [Retry 1/5].


In [6]:
from langchain_community.document_loaders import PyPDFLoader #specifically for reading PDFs.
# directory loader
loader = DirectoryLoader(  #Create a loader that looks inside a directory and loads files
    path=docs_dir_path,  #path of directory
    glob="./*.pdf",  #only look for files with .pdf extension
    loader_cls=PyPDFLoader  #use this to read the pdf
)

# single file can also be loaded with UnstructuredFileLoader

In [7]:
documents = loader.load() #This is where the actual loading happens.

In [8]:
len(documents)

8

In [9]:
#initializing the text splitter
text_splitter=CharacterTextSplitter(
    chunk_size=2000,  #Try to make each chunk around 2000 characters.
    chunk_overlap=500)  #When a single Document needs to be split into multiple chunks, those chunks can share 500 characters.

In [10]:
#spllting text itno chunks
text_chunks=text_splitter.split_documents(documents)

In [11]:
len(text_chunks)

7

In [12]:
#here as different pages are divided into doxuments so each doucmnet contains less words therefore 1 document=1chunk
print(text_chunks[0])
print('-------------------------------------------------')
print(text_chunks[1])

page_content='Experiment 1 
 
Student Name: Sejal UID: 24BAI70050 
Branch: CSE - AIML Section/Group: 24AIT_KRG G1 
Semester: 5 Date of Performance: 
8/7/26 Subject Name: Soft Computing Subject Code: 24CSH-339 
 
 
Aim 
Getting started with the Python 3.x and installing libraries of Tensorflow, Keras, 
Pytorch. 
 
Platform 
Anaconda Navigator 
Theory 
Anaconda is an open-source distribution for python and R. It is used for data 
science, machine  learning, deep learning, etc. With the availability of more than 
300 libraries for  data science, it becomes fairly optimal for any programmer to 
work on anaconda for data science. Anaconda helps in simplified package 
management and deployment.  Anaconda comes with a wide variety of tools to 
easily collect data from various  sources using various machine learning and AI 
algorithms. It helps in getting an easily manageable environment setup which can 
deploy any project with the click of a single button. 
 
Procedure 
Installation of Anacon

In [13]:
#creating the vector db
vector_store=Chroma.from_documents(  #Take these documents, create embeddings for them, and put them into Chroma.
documents=text_chunks,
embedding=embedding,
persist_directory=vector_db_path,
collection_name=collection_name
)